In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mdrashidshahariar/mit-bihscd-and-normal-sinus-rhythm-data/ECG_MIT-BIH_SIN_Holter/README.md
/kaggle/input/datasets/mdrashidshahariar/mit-bihscd-and-normal-sinus-rhythm-data/ECG_MIT-BIH_SIN_Holter/nsrdb_250hz/18184.dat
/kaggle/input/datasets/mdrashidshahariar/mit-bihscd-and-normal-sinus-rhythm-data/ECG_MIT-BIH_SIN_Holter/nsrdb_250hz/19090.dat
/kaggle/input/datasets/mdrashidshahariar/mit-bihscd-and-normal-sinus-rhythm-data/ECG_MIT-BIH_SIN_Holter/nsrdb_250hz/18177.dat
/kaggle/input/datasets/mdrashidshahariar/mit-bihscd-and-normal-sinus-rhythm-data/ECG_MIT-BIH_SIN_Holter/nsrdb_250hz/16773.dat
/kaggle/input/datasets/mdrashidshahariar/mit-bihscd-and-normal-sinus-rhythm-data/ECG_MIT-BIH_SIN_Holter/nsrdb_250hz/19830.dat
/kaggle/input/datasets/mdrashidshahariar/mit-bihscd-and-normal-sinus-rhythm-data/ECG_MIT-BIH_SIN_Holter/nsrdb_250hz/16786.dat
/kaggle/input/datasets/mdrashidshahariar/mit-bihscd-and-normal-sinus-rhythm-data/ECG_MIT-BIH_SIN_Holter/nsrdb_250hz/19090.hea
/kag

In [2]:
pip install wfdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 1.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
# ================================================
# Implementation of the Paper: Early SCD Prediction
# ================================================

import os
import numpy as np
import pandas as pd
import wfdb
import matplotlib.pyplot as plt
from tqdm import tqdm
import scipy.signal as signal
from scipy.signal import resample
import pywt
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries imported successfully!")
print("Dataset path:", "/kaggle/input/")

Libraries imported successfully!
Dataset path: /kaggle/input/


In [4]:
import os
import wfdb

ROOT = "/kaggle/input/datasets/mdrashidshahariar/mit-bihscd-and-normal-sinus-rhythm-data/ECG_MIT-BIH_SIN_Holter"

SCD_PATH = os.path.join(ROOT, "sudden-cardiac-death-holter-database-1.0.0")

records = sorted([
    f[:-4]
    for f in os.listdir(SCD_PATH)
    if f.endswith(".hea")
])

print(records)
print("Total records:", len(records))

['30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52']
Total records: 23


In [5]:
records_file = os.path.join(SCD_PATH, "RECORDS")

with open(records_file, "r") as f:
    records_from_file = [line.strip() for line in f]

print("Records in RECORDS file:")
print(records_from_file)
print("\nTotal:", len(records_from_file))

Records in RECORDS file:
['30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52']

Total: 23


In [6]:
annotators = []

with open(os.path.join(SCD_PATH, "ANNOTATORS")) as f:
    for line in f:
        line = line.strip()
        if line:
            annotators.append(line.split()[0])

print(annotators)

['ari', 'atr']


In [7]:
import os

for rec in records:

    print(f"\n===== Record {rec} =====")

    for ext in annotators:

        filename = os.path.join(SCD_PATH, rec + "." + ext)

        print(f"{ext:5s} ->", os.path.exists(filename))


===== Record 30 =====
ari   -> True
atr   -> True

===== Record 31 =====
ari   -> True
atr   -> True

===== Record 32 =====
ari   -> True
atr   -> True

===== Record 33 =====
ari   -> True
atr   -> False

===== Record 34 =====
ari   -> True
atr   -> True

===== Record 35 =====
ari   -> True
atr   -> True

===== Record 36 =====
ari   -> True
atr   -> True

===== Record 37 =====
ari   -> True
atr   -> False

===== Record 38 =====
ari   -> True
atr   -> False

===== Record 39 =====
ari   -> True
atr   -> False

===== Record 40 =====
ari   -> True
atr   -> False

===== Record 41 =====
ari   -> True
atr   -> True

===== Record 42 =====
ari   -> True
atr   -> False

===== Record 43 =====
ari   -> True
atr   -> False

===== Record 44 =====
ari   -> True
atr   -> False

===== Record 45 =====
ari   -> True
atr   -> True

===== Record 46 =====
ari   -> True
atr   -> True

===== Record 47 =====
ari   -> True
atr   -> False

===== Record 48 =====
ari   -> True
atr   -> False

===== Record 49 ====

In [8]:
for rec in ["30","31","34","35","49"]:

    print("="*80)
    print(rec)

    filename = os.path.join(SCD_PATH,f"{rec}.hea-")

    if os.path.exists(filename):

        with open(filename) as f:

            print(f.read())

30
30 2 250 22099250 12:00:00
30.dat 212 800 12 0 51 -24065 0 record 30, signal 0
30.dat 212 800 12 0 145 21051 0 record 30, signal 1
#Produced by xform_new from record 30, beginning at 26:35.000
#vfon: 07:54:33

31
31 2 250 12580000 10:15:00
31.dat 212 800 12 0 -8 -20354 0 record 31, signal 0 
31.dat 212 800 12 0 -16 8135 0 record 31, signal 1 
#Produced by xform_new from record 31, beginning at 13:20.000 
#vfon: 13:42:24 

34
34 2 250 6380000 12:00:00
34.dat 212 800 12 0 -432 3942 0 record 34, signal 0 
34.dat 212 800 12 0 15 -656 0 record 34, signal 1 
#Produced by xform_new from record 34, beginning at 0 
#vfon: 06:35:44 

35
35 2 250 22380000 12:00:00
35.dat 212 800 12 0 -225 -29662 0 record 35, signal 0 
35.dat 212 800 12 0 198 -2153 0 record 35, signal 1 
#Produced by xform_new from record 35, beginning at 0 
#vfon: 24:34:56

49
49 2 250 22525000 13:40:00
49.dat 212 800 12 0 -76 13739 0 record 49, signal 0 
49.dat 212 800 12 0 -77 16736 0 record 49, signal 1 
#Produced by xform 

In [9]:
import os
import re
import pandas as pd

records = [str(i) for i in range(30, 53)]

vf_info = []

for rec in records:

    hea_dash = os.path.join(SCD_PATH, f"{rec}.hea-")

    if not os.path.exists(hea_dash):
        continue

    with open(hea_dash, "r") as f:
        text = f.read()

    # Extract recording start time
    first_line = text.splitlines()[0]
    start_time = first_line.split()[-1]

    # Extract VF onset
    match = re.search(r"#vfon:\s*([0-9:]+)", text)

    if match:
        vf_time = match.group(1)
    else:
        vf_time = None

    vf_info.append({
        "Record": rec,
        "Recording_Start": start_time,
        "VF_Onset": vf_time
    })

vf_df = pd.DataFrame(vf_info)

vf_df

,Record,Recording_Start,VF_Onset
0,30,12:00:00,07:54:33
1,31,10:15:00,13:42:24
2,32,05:04:00,16:45:18
3,33,10:38:00,04:46:19
4,34,12:00:00,06:35:44
5,35,12:00:00,24:34:56
6,36,15:09:00,18:59:01
7,37,14:29:00,01:31:13
8,38,12:00:00,08:01:54
9,39,12:00:00,04:37:51


In [10]:
no_vf = vf_df[vf_df["VF_Onset"].isna()]

print(no_vf)

   Record Recording_Start VF_Onset
10     40        09:50:00     None
12     42        12:00:00     None
19     49        13:40:00     None


In [11]:
usable_df = vf_df[vf_df["VF_Onset"].notna()].copy()

print("Usable SCD Patients:", len(usable_df))
usable_df

Usable SCD Patients: 20


,Record,Recording_Start,VF_Onset
0,30,12:00:00,07:54:33
1,31,10:15:00,13:42:24
2,32,05:04:00,16:45:18
3,33,10:38:00,04:46:19
4,34,12:00:00,06:35:44
5,35,12:00:00,24:34:56
6,36,15:09:00,18:59:01
7,37,14:29:00,01:31:13
8,38,12:00:00,08:01:54
9,39,12:00:00,04:37:51


In [12]:
usable_df.to_csv("usable_scd_patients.csv", index=False)

print("Saved usable_scd_patients.csv")

Saved usable_scd_patients.csv


In [13]:
import os
import wfdb
import pandas as pd

NSR_PATH = "/kaggle/input/datasets/mdrashidshahariar/mit-bihscd-and-normal-sinus-rhythm-data/ECG_MIT-BIH_SIN_Holter/nsrdb_250hz"

records = sorted([
    f[:-4]
    for f in os.listdir(NSR_PATH)
    if f.endswith(".hea")
])

info = []

for rec in records:

    record = wfdb.rdrecord(os.path.join(NSR_PATH, rec))

    fs = record.fs
    samples = len(record.p_signal)
    minutes = samples / fs / 60

    info.append({
        "Record": rec,
        "Samples": samples,
        "Minutes": round(minutes,2)
    })

df = pd.DataFrame(info)

print(df)
print("\nShortest recording:", df["Minutes"].min())
print("Longest recording :", df["Minutes"].max())

   Record   Samples  Minutes
0   16265  22912000  1527.47
1   16272  22500000  1500.00
2   16273  22176000  1478.40
3   16420  21584000  1438.93
4   16483  23360000  1557.33
5   16539  22124000  1474.93
6   16773  21576000  1438.40
7   16786  22040000  1469.33
8   16795  21224000  1414.93
9   17052  20820000  1388.00
10  17453  21944000  1462.93
11  18177  23360000  1557.33
12  18184  21372000  1424.80
13  19088  21420000  1428.00
14  19090  21764000  1450.93
15  19093  20910000  1394.00
16  19140  21756000  1450.40
17  19830  20902000  1393.47

Shortest recording: 1388.0
Longest recording : 1557.33


In [14]:
import os
import wfdb
import pandas as pd

metadata = []

for _, row in vf_df.iterrows():

    rec = str(row["Record"])

    record = wfdb.rdrecord(os.path.join(SCD_PATH, rec))

    samples = len(record.p_signal)
    fs = record.fs
    duration_sec = samples / fs
    duration_min = duration_sec / 60

    metadata.append({
        "Record": rec,
        "Recording_Start": row["Recording_Start"],
        "VF_Onset": row["VF_Onset"],
        "Sampling_Rate": fs,
        "Signal_Length": samples,
        "Duration_Min": round(duration_min,2)
    })

metadata_df = pd.DataFrame(metadata)

metadata_df

,Record,Recording_Start,VF_Onset,Sampling_Rate,Signal_Length,Duration_Min
0,30,12:00:00,07:54:33,250,22099250,1473.28
1,31,10:15:00,13:42:24,250,12580000,838.67
2,32,05:04:00,16:45:18,250,21900000,1460.00
3,33,10:38:00,04:46:19,250,22095000,1473.00
4,34,12:00:00,06:35:44,250,6380000,425.33
5,35,12:00:00,24:34:56,250,22380000,1492.00
6,36,15:09:00,18:59:01,250,18320000,1221.33
7,37,14:29:00,01:31:13,250,22620000,1508.00
8,38,12:00:00,08:01:54,250,16476250,1098.42
9,39,12:00:00,04:37:51,250,5205000,347.00


In [15]:
def time_to_seconds(t):

    if pd.isna(t):
        return None

    h, m, s = map(int, t.split(":"))

    return h*3600 + m*60 + s

metadata_df["VF_Onset_Seconds"] = metadata_df["VF_Onset"].apply(time_to_seconds)

metadata_df

,Record,Recording_Start,VF_Onset,Sampling_Rate,Signal_Length,Duration_Min,VF_Onset_Seconds
0,30,12:00:00,07:54:33,250,22099250,1473.28,28473.0
1,31,10:15:00,13:42:24,250,12580000,838.67,49344.0
2,32,05:04:00,16:45:18,250,21900000,1460.00,60318.0
3,33,10:38:00,04:46:19,250,22095000,1473.00,17179.0
4,34,12:00:00,06:35:44,250,6380000,425.33,23744.0
5,35,12:00:00,24:34:56,250,22380000,1492.00,88496.0
6,36,15:09:00,18:59:01,250,18320000,1221.33,68341.0
7,37,14:29:00,01:31:13,250,22620000,1508.00,5473.0
8,38,12:00:00,08:01:54,250,16476250,1098.42,28914.0
9,39,12:00:00,04:37:51,250,5205000,347.00,16671.0


In [16]:
metadata_df["Usable"] = metadata_df["VF_Onset"].notna()

metadata_df

,Record,Recording_Start,VF_Onset,Sampling_Rate,Signal_Length,Duration_Min,VF_Onset_Seconds,Usable
0,30,12:00:00,07:54:33,250,22099250,1473.28,28473.0,True
1,31,10:15:00,13:42:24,250,12580000,838.67,49344.0,True
2,32,05:04:00,16:45:18,250,21900000,1460.00,60318.0,True
3,33,10:38:00,04:46:19,250,22095000,1473.00,17179.0,True
4,34,12:00:00,06:35:44,250,6380000,425.33,23744.0,True
5,35,12:00:00,24:34:56,250,22380000,1492.00,88496.0,True
6,36,15:09:00,18:59:01,250,18320000,1221.33,68341.0,True
7,37,14:29:00,01:31:13,250,22620000,1508.00,5473.0,True
8,38,12:00:00,08:01:54,250,16476250,1098.42,28914.0,True
9,39,12:00:00,04:37:51,250,5205000,347.00,16671.0,True


In [17]:
usable_scd = metadata_df[metadata_df["Usable"]].copy()

usable_scd.reset_index(drop=True, inplace=True)

usable_scd

,Record,Recording_Start,VF_Onset,Sampling_Rate,Signal_Length,Duration_Min,VF_Onset_Seconds,Usable
0,30,12:00:00,07:54:33,250,22099250,1473.28,28473.0,True
1,31,10:15:00,13:42:24,250,12580000,838.67,49344.0,True
2,32,05:04:00,16:45:18,250,21900000,1460.00,60318.0,True
3,33,10:38:00,04:46:19,250,22095000,1473.00,17179.0,True
4,34,12:00:00,06:35:44,250,6380000,425.33,23744.0,True
5,35,12:00:00,24:34:56,250,22380000,1492.00,88496.0,True
6,36,15:09:00,18:59:01,250,18320000,1221.33,68341.0,True
7,37,14:29:00,01:31:13,250,22620000,1508.00,5473.0,True
8,38,12:00:00,08:01:54,250,16476250,1098.42,28914.0,True
9,39,12:00:00,04:37:51,250,5205000,347.00,16671.0,True


In [18]:
excluded = metadata_df[~metadata_df["Usable"]]

excluded

,Record,Recording_Start,VF_Onset,Sampling_Rate,Signal_Length,Duration_Min,VF_Onset_Seconds,Usable
10,40,09:50:00,None,250,22395000,1493.00,NaN,False
12,42,12:00:00,None,250,22622500,1508.17,NaN,False
19,49,13:40:00,None,250,22380957,1492.06,NaN,False


In [19]:
metadata_df.to_csv("scd_metadata.csv", index=False)

usable_scd.to_csv("usable_scd_patients.csv", index=False)

excluded.to_csv("excluded_patients.csv", index=False)

print("Saved successfully.")

Saved successfully.


In [20]:
nsr_metadata = df.copy()

nsr_metadata.rename(columns={
    "Record":"NSR_Record",
    "Samples":"Signal_Length",
    "Minutes":"Duration_Min"
}, inplace=True)

nsr_metadata["Sampling_Rate"] = 250

nsr_metadata.to_csv("nsr_metadata.csv", index=False)

nsr_metadata

,NSR_Record,Signal_Length,Duration_Min,Sampling_Rate
0,16265,22912000,1527.47,250
1,16272,22500000,1500.00,250
2,16273,22176000,1478.40,250
3,16420,21584000,1438.93,250
4,16483,23360000,1557.33,250
5,16539,22124000,1474.93,250
6,16773,21576000,1438.40,250
7,16786,22040000,1469.33,250
8,16795,21224000,1414.93,250
9,17052,20820000,1388.00,250


In [21]:
print("="*50)

print("Usable SCD :", len(usable_scd))

print("Excluded SCD :", len(excluded))

print("NSR Subjects :", len(nsr_metadata))

print("="*50)

print("SCD Sampling Rate :", usable_scd["Sampling_Rate"].unique())

print("NSR Sampling Rate :", nsr_metadata["Sampling_Rate"].unique())

Usable SCD : 20
Excluded SCD : 3
NSR Subjects : 18
SCD Sampling Rate : [250]
NSR Sampling Rate : [250]


In [22]:
import os

os.makedirs("/kaggle/working/metadata", exist_ok=True)

usable_scd.to_csv(
    "/kaggle/working/metadata/usable_scd_patients.csv",
    index=False
)

metadata_df.to_csv(
    "/kaggle/working/metadata/scd_metadata.csv",
    index=False
)

excluded.to_csv(
    "/kaggle/working/metadata/excluded_patients.csv",
    index=False
)

nsr_metadata.to_csv(
    "/kaggle/working/metadata/nsr_metadata.csv",
    index=False
)

In [23]:
#Summay
print("="*60)
print("DATABASE SUMMARY")
print("="*60)

print(f"Original SCD Records : {len(metadata_df)}")
print(f"Usable SCD Records   : {len(usable_scd)}")
print(f"Excluded SCD Records : {len(excluded)}")
print(f"NSR Subjects         : {len(nsr_metadata)}")

print()

print("Excluded:", excluded["Record"].tolist())

print()

print("Sampling Rate SCD :", usable_scd["Sampling_Rate"].unique())
print("Sampling Rate NSR :", nsr_metadata["Sampling_Rate"].unique())

print("="*60)

DATABASE SUMMARY
Original SCD Records : 23
Usable SCD Records   : 20
Excluded SCD Records : 3
NSR Subjects         : 18

Excluded: ['40', '42', '49']

Sampling Rate SCD : [250]
Sampling Rate NSR : [250]


In [24]:
print("Notebook Version : 1.0")
print("Random Seed      : 42")
print("Date             : 2026-06-30")

Notebook Version : 1.0
Random Seed      : 42
Date             : 2026-06-30
